# GPT Action 库：SQL 数据库


## 简介

本指南面向希望通过 GPT Action 让 ChatGPT 具备查询 SQL 数据库能力的开发者。阅读本指南之前，请先熟悉以下内容：



* [GPT Actions 简介](https://platform.openai.com/docs/actions)
* [GPT Actions 库简介](https://platform.openai.com/docs/actions/actions-library)
* [从零构建 GPT Action 的示例](https://platform.openai.com/docs/actions/getting-started)

本指南概述了通过中间件应用将 ChatGPT 连接到 SQL 数据库所需的工作流程。本示例使用 PostgreSQL 数据库，但对于所有 SQL 数据库（MySQL、MS SQL Server、Amazon Aurora、Google Cloud 上的 SQL Server 等），流程应该都是类似的。本文档概述了创建 GPT Action 所需的步骤，该 Action 可以：



* 对 SQL 数据库执行读取查询
* 通过文本响应返回记录
* 通过 CSV 文件返回记录


### 价值与业务用例示例

**价值**：用户现在可以利用 ChatGPT 的自然语言能力来回答关于 SQL 数据库中数据的问题：



* 业务用户无需编写 SQL 或向分析师提交请求，即可获取 SQL 数据库中的信息
* 数据分析师可以提取数据并用 ChatGPT 进行分析，完成超出 SQL 查询能力范围的复杂分析

**用例示例**：



* 业务用户需要回答有关其销售漏斗的问题
* 数据分析师需要对大型数据集进行回归分析


## 应用设计考量

由于大多数托管 SQL 数据库并不提供用于提交查询的 REST API，你需要一个中间件应用来执行以下功能：



1. 通过 REST API 请求接收数据库查询
2. 将查询转发给集成的 SQL 数据库
3. 将数据库响应转换为 CSV 文件
4. 将 CSV 文件返回给请求方

设计第一个功能主要有两种方式：



1. 中间件提供单一方法，接收由 GPT 生成的任意 SQL 查询并将其转发给数据库。这种方式的好处包括：
    1. 易于开发
    2. 灵活（无需预先设想用户会发起哪些类型的查询）
    3. 维护成本低（无需随数据库变更而更新 API schema）
2. 中间件提供多个方法，分别对应特定的允许查询。这种方式的好处包括：
    4. 更强的控制力
    5. 模型在生成 SQL 时出错的机会更少

本指南将聚焦于方案 1。对方案 2 感兴趣的读者，可以考虑使用 [PostgREST](https://github.com/PostgREST/postgrest) 或 [Hasura](https://hasura.io/) 之类的服务来简化流程。

![描述用户、GPT、中间件和数据库之间交互的应用架构图](../../../images/gptactions_sql_database_middleware.png)
_应用架构图_

## 中间件考量

开发者既可以构建自定义中间件（通常以无服务器函数的形式部署在 AWS、GCP 或 MS Azure 等云服务提供商上），也可以使用第三方解决方案（如 [Mulesoft Anypoint](https://www.mulesoft.com/platform/enterprise-integration) 或 [Retool Workflows](https://retool.com/products/workflows)）。使用第三方中间件可以加快开发进程，但灵活性不如自行构建。

自行构建中间件可以让你更好地控制应用的行为。自定义中间件的示例请参阅我们的 [Azure Functions cookbook](https://cookbook.openai.com/examples/chatgpt/gpt_actions_library/gpt_middleware_azure_function)。

本指南不会深入中间件设置的细节，而是聚焦于中间件与 GPT 以及 SQL 数据库之间的接口。


## 工作流步骤


### 1) GPT 生成 SQL 查询

GPT 非常擅长根据用户的自然语言提示编写 SQL 查询。你可以通过以下任一方式让 GPT 获取数据库 schema，从而提升其生成查询的能力：



1. 指示 GPT 首先查询数据库以获取 schema（我们的 [BigQuery cookbook](https://cookbook.openai.com/examples/chatgpt/gpt_actions_library/gpt_action_bigquery#custom-gpt-instructions) 中对这种方式有更详细的演示）。
2. 在 GPT 指令中直接提供 schema（最适合小型、静态的 schema）

下面是包含一个简单数据库 schema 信息的 GPT 指令示例：


In [ ]:
# Context
You are a data analyst. Your job is to assist users with their business questions by analyzing the data contained in a PostgreSQL database.

## Database Schema

### Accounts Table
**Description:** Stores information about business accounts.

| Column Name  | Data Type      | Constraints                        | Description                             |
|--------------|----------------|------------------------------------|-----------------------------------------|
| account_id   | INT            | PRIMARY KEY, AUTO_INCREMENT, NOT NULL | Unique identifier for each account      |
| account_name | VARCHAR(255)   | NOT NULL                           | Name of the business account            |
| industry     | VARCHAR(255)   |                                    | Industry to which the business belongs  |
| created_at   | TIMESTAMP      | NOT NULL, DEFAULT CURRENT_TIMESTAMP | Timestamp when the account was created  |

### Users Table
**Description:** Stores information about users associated with the accounts.

| Column Name  | Data Type      | Constraints                        | Description                             |
|--------------|----------------|------------------------------------|-----------------------------------------|
| user_id      | INT            | PRIMARY KEY, AUTO_INCREMENT, NOT NULL | Unique identifier for each user         |
| account_id   | INT            | NOT NULL, FOREIGN KEY (References Accounts(account_id)) | Foreign key referencing Accounts(account_id) |
| username     | VARCHAR(50)    | NOT NULL, UNIQUE                   | Username chosen by the user             |
| email        | VARCHAR(100)   | NOT NULL, UNIQUE                   | User's email address                    |
| role         | VARCHAR(50)    |                                    | Role of the user within the account     |
| created_at   | TIMESTAMP      | NOT NULL, DEFAULT CURRENT_TIMESTAMP | Timestamp when the user was created     |

### Revenue Table
**Description:** Stores revenue data related to the accounts.

| Column Name  | Data Type      | Constraints                        | Description                             |
|--------------|----------------|------------------------------------|-----------------------------------------|
| revenue_id   | INT            | PRIMARY KEY, AUTO_INCREMENT, NOT NULL | Unique identifier for each revenue record |
| account_id   | INT            | NOT NULL, FOREIGN KEY (References Accounts(account_id)) | Foreign key referencing Accounts(account_id) |
| amount       | DECIMAL(10, 2) | NOT NULL                           | Revenue amount                          |
| revenue_date | DATE           | NOT NULL                           | Date when the revenue was recorded      |

# Instructions:
1. When the user asks a question, consider what data you would need to answer the question and confirm that the data should be available by consulting the database schema.
2. Write a PostgreSQL-compatible query and submit it using the `databaseQuery` API method.
3. Use the response data to answer the user's question.
4. If necessary, use code interpreter to perform additional analysis on the data until you are able to answer the user's question.

### 2) GPT 将 SQL 查询发送给中间件

为了让 GPT 与中间件通信，我们将配置一个 GPT Action。中间件需要提供一个接受 SQL 查询字符串的 REST API 端点。你可以用多种方式设计这个接口。下面是一个简单端点的 OpenAPI schema 示例，它在 POST 操作中接受一个“q”参数：

In [ ]:
openapi: 3.1.0
info:
  title: PostgreSQL API
  description: API for querying a PostgreSQL database
  version: 1.0.0
servers:
  - url: https://my.middleware.com/v1
    description: middleware service
paths:
  /api/query:
    post:
      operationId: databaseQuery
      summary: Query a PostgreSQL database
      requestBody:
        required: true
        content:
          application/json:
            schema:
              type: object
              properties:
                q:
                  type: string
                  example: select * from users
      responses:
        "200":
          description: database records
          content:
            application/json:
              schema:
                type: object
                properties:
                  openaiFileResponse:
                    type: array
                    items:
                      type: object
                      properties:
                        name:
                          type: string
                          description: The name of the file.
                        mime_type:
                          type: string
                          description: The MIME type of the file.
                        content:
                          type: string
                          format: byte
                          description: The content of the file in base64 encoding.
        "400":
          description: Bad Request. Invalid input.
        "401":
          description: Unauthorized. Invalid or missing API key.
      security:
        - ApiKey: []
components:
  securitySchemes:
    ApiKey:
      type: apiKey
      in: header
      name: X-Api-Key
  schemas: {}

**关于身份验证的说明：** 上例中的 API 接口接受一个系统级 API 密钥，该密钥与 GPT 的配置一同存储，并用于对所有 GPT 用户的请求进行身份验证。GPT Actions 也支持 OAuth 身份验证，可实现用户级的身份验证和授权。[详细了解 GPT Action 身份验证选项](https://platform.openai.com/docs/actions/authentication)。

由于用户是向中间件而非直接向底层数据库进行身份验证，强制实施用户级访问控制（表级或行级权限）需要更多工作。不过，如果 GPT 的用户对底层数据库拥有不同级别的访问权限，就可能需要这样做。

 为了强制实施用户级权限，你的中间件应当：



1. 接收 IdP 在 OAuth 流程中提供的用户元数据，并从中提取其身份信息
2. 查询数据库以获取该用户的数据库权限
3. 向数据库发出命令，在会话的剩余时间内强制执行相应权限

为了保持良好的用户体验，你需要为每个用户动态获取其可用的数据库 schema，而不是将 schema 数据直接写进 GPT 指令中。这样可以确保 GPT 只能访问它能代表当前用户查询的表。


### 3) 中间件将 SQL 查询转发给数据库 

你的中间件需要实现一个数据库驱动或客户端库，以便直接查询 PostgreSQL 数据库。如果你使用第三方中间件，中间件供应商应当提供 SQL 数据库的原生连接器。如果你自行构建中间件，可能需要接入数据库供应商或第三方提供的客户端库。例如，这里有一份由社区维护的 PostgreSQL 客户端库列表：[https://wiki.postgresql.org/wiki/List_of_drivers](https://wiki.postgresql.org/wiki/List_of_drivers)

在这一步中，中间件应用需要从 GPT 发来的请求中提取 SQL 字符串，并使用客户端库提供的方法将其转发给数据库。

**关于只读权限的说明：** 由于这种设计模式会让你的数据库处理任意由 AI 生成的 SQL 查询，你应确保中间件应用对数据库只有只读权限。这样可以保证 AI 生成的查询无法插入新数据或修改现有数据。如果你的用例需要写入权限，请考虑部署针对特定操作的端点，而不是接受任意 SQL。


### 4) 数据库将记录返回给中间件

根据你所使用的客户端库，中间件可能会以多种格式接收记录。一种常见模式是中间件接收一个 JSON 对象数组，每个对象代表一条与查询匹配的数据库记录：


In [ ]:
[
  {
    "account_id": 1,
    "number_of_users": 10,
    "total_revenue": 43803.96,
    "revenue_per_user": 4380.40
  },
  {
    "account_id": 2,
    "number_of_users": 12,
    "total_revenue": 77814.84,
    "revenue_per_user": 6484.57
  },
  ...
]

### 5) 中间件将记录转换为 base64 编码的 CSV 文件

为了让 ChatGPT 能够分析大量记录，它需要获取 CSV 格式的数据。GPT Actions 接口允许 GPT [接收 base64 编码的文件](https://platform.openai.com/docs/actions/sending-files/returning-files)，大小上限为 10mb。

你的中间件需要执行两个操作：


#### 将记录转换为 CSV 格式

许多编程语言都提供处理 CSV 文件的原生库（例如 Python 的 [csv](https://docs.python.org/3/library/csv.html) 库）。

下面是一个中间件将 JSON 对象数组转换为 CSV 文件的示例：

In [ ]:
import json
import csv

# Sample JSON array of objects
json_data = '''
[
    {"account_id": 1, "number_of_users": 10, "total_revenue": 43803.96, "revenue_per_user": 4380.40}, 
    {"account_id": 2, "number_of_users": 12, "total_revenue": 77814.84, "revenue_per_user": 6484.57}
]
'''

# Load JSON data
data = json.loads(json_data)

# Define the CSV file name
csv_file = 'output.csv'

# Write JSON data to CSV
with open(csv_file, 'w', newline='') as csvfile:
    # Create a CSV writer object
    csvwriter = csv.writer(csvfile)
    
    # Write the header (keys of the first dictionary)
    header = data[0].keys()
    csvwriter.writerow(header)
    
    # Write the data rows
    for row in data:
        csvwriter.writerow(row.values())

print(f"JSON data has been written to {csv_file}")

#### 对 CSV 文件进行 base64 编码

许多编程语言都提供处理 base64 编码的原生库（例如 Python 的 [base64](https://docs.python.org/3/library/base64.html) 库）。

下面是一个中间件对上一步生成的 CSV 文件进行 base64 编码的示例：

In [ ]:
import base64 

# Base64 encode the CSV file
encoded_string = base64.b64encode(open('output.csv', 'rb').read()).decode('utf-8')

print("Base64 Encoded CSV:")
print(encoded_string)

### 6) 中间件将 base64 编码的 CSV 文件返回给 GPT

为了让 GPT Actions 接口能够处理 base64 编码的 CSV 文件，中间件返回的响应中必须包含一个 `openaiFileResponse` 参数。提供的值必须是文件对象数组或文件链接数组（详见 [Actions 文档](https://platform.openai.com/docs/actions/sending-files/returning-files)）。在本示例中，我们使用文件对象数组。

下面是一个有效响应体的示例：

In [ ]:
{
  "openaiFileResponse": [
    {
      "name": "output.csv",
      "mime_type": "text/csv",
      "content": "ImFjY291bn...NC41NyI="
    }
  ]
}

### 7) GPT 处理返回的文件

GPT 收到 base64 编码的 CSV 文件后，会自动解码并处理该文件，以回答用户的问题。这可能涉及使用 [Code Interpreter 对 CSV 文件进行进一步分析](https://help.openai.com/en/articles/9213685-extracting-insights-with-chatgpt-data-analysis)，其过程与用户通过提示上传 CSV 文件时完全相同。

**注意：** 如果你希望能对返回的文件进行进一步分析，必须在 GPT 中启用 _Code Interpreter & Data Analysis_ 功能。


## 总结

GPT Actions 为从 SQL 数据库等外部数据源获取数据提供了一个灵活的框架。让 ChatGPT 具备查询数据库的能力，可以大幅扩展它作为知识助手和分析师的能力。

_有希望我们优先支持的集成吗？我们的集成中有错误吗？请在我们的 GitHub 中提交 PR 或 issue，我们会及时查看。_